In [ ]:
# Step 1: Setup and Data Preparation
!pip install torch transformers torchvision pycocotools fastapi uvicorn onnx onnxruntime

from pycocotools.coco import COCO
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer
import torch.nn as nn
import torch.optim as optim
import torch
import os

# COCO Dataset Configuration
data_dir = './coco'
ann_file = os.path.join(data_dir, 'annotations/captions_train2017.json')
img_dir = os.path.join(data_dir, 'train2017')

# Download COCO dataset (first 10k images for demonstration)
!mkdir -p {data_dir}
!wget http://images.cocodataset.org/zips/train2017.zip -P {data_dir}
!wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip -P {data_dir}
!unzip -q {data_dir}/train2017.zip -d {data_dir}
!unzip -q {data_dir}/annotations_trainval2017.zip -d {data_dir}

In [ ]:
# Step 2: Model Architecture
class ImageCaptioningModel(nn.Module):
    def __init__(self, embed_size=256, hidden_size=512, vocab_size=30522):
        super().__init__()
        # CNN Encoder (ResNet-50)
        resnet = torch.hub.load('pytorch/vision:v0.10.0', 'resnet50', pretrained=True)
        self.encoder = nn.Sequential(*list(resnet.children())[:-1])
        self.feature_proj = nn.Linear(2048, embed_size)  # Project ResNet features to embed_size

        # Transformer Decoder
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.positional_encoding = nn.Parameter(torch.zeros(1, 50, embed_size))  # Max caption length=50
        self.transformer = nn.Transformer(d_model=embed_size, nhead=8,
                                         num_encoder_layers=3,
                                         num_decoder_layers=3)
        self.fc = nn.Linear(embed_size, vocab_size)

    def forward(self, images, captions):
        # Image encoding (batch_size, 2048)
        features = self.encoder(images).squeeze()
        # Project to embed_size (batch_size, embed_size)
        features_proj = self.feature_proj(features)

        # Add sequence dimension for transformer (1, batch_size, embed_size)
        encoder_output = features_proj.unsqueeze(0)

        # Caption processing (batch_size, seq_len) -> (seq_len, batch_size, embed_size)
        embedded = self.embedding(captions) + self.positional_encoding[:, :captions.size(1), :]
        decoder_input = embedded.permute(1, 0, 2)

        # Transformer forward pass
        transformer_output = self.transformer(encoder_output, decoder_input)
        return self.fc(transformer_output.permute(1, 0, 2))

In [ ]:
# Step 3: Data Pipeline
class COCODataset(Dataset):
    def __init__(self, root, ann_file, transform=None, max_length=50, vocab_size=30522):
        self.root = root
        self.coco = COCO(ann_file)
        self.ids = list(self.coco.anns.keys())
        self.transform = transform
        self.tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        self.max_length = max_length
        self.vocab_size = vocab_size

    def __getitem__(self, index):
        ann_id = self.ids[index]
        caption = self.coco.anns[ann_id]['caption']
        img_id = self.coco.anns[ann_id]['image_id']
        path = self.coco.loadImgs(img_id)[0]['file_name']

        image = Image.open(os.path.join(self.root, path)).convert('RGB')
        if self.transform:
            image = self.transform(image)

        # Tokenize and pad/truncate captions
        tokens = self.tokenizer(caption, padding='max_length',
                                max_length=self.max_length, truncation=True,
                                return_tensors='pt')['input_ids'].squeeze()

        # Ensure all tokens are within vocab size
        tokens = torch.clamp(tokens, 0, self.vocab_size - 1)

        return image, tokens

    def __len__(self):
        return len(self.ids)

In [ ]:
# Step 4: Training Loop
# Define device and model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize model and move it to the device
model = ImageCaptioningModel(embed_size=256, hidden_size=512, vocab_size=30522).to(device)

# Loss function and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding token (BERT pads with 0)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Training loop
for epoch in range(5):  # Reduced epochs for demonstration
    model.train()
    total_loss = 0
    for images, captions in dataloader:
        # Move data to the device
        images, captions = images.to(device), captions.to(device)

        # Forward pass (teacher forcing)
        outputs = model(images, captions[:, :-1])  # Input captions shifted by 1
        targets = captions[:, 1:]  # Target captions shifted by 1

        # Compute loss
        loss = criterion(outputs.reshape(-1, 30522), targets.reshape(-1))  # Reshape for loss calculation

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f'Epoch {epoch+1}, Loss: {total_loss/len(dataloader):.4f}')

In [ ]:
# Step 5: ONNX Optimization
dummy_image = torch.randn(1, 3, 224, 224).to(device)
dummy_caption = torch.randint(0, 10000, (1, 49)).to(device)
torch.onnx.export(model, (dummy_image, dummy_caption), "caption_model.onnx",
                  input_names=['image', 'caption'], output_names=['output'])

# Step 6: FastAPI Deployment
from fastapi import FastAPI, File, UploadFile
from PIL import Image
import io

app = FastAPI()
ort_session = ort.InferenceSession("caption_model.onnx")

@app.post("/generate-caption/")
async def generate_caption(file: UploadFile = File(...)):
    image = Image.open(io.BytesIO(await file.read())).convert('RGB')
    image = transform(image).unsqueeze(0).numpy()
    outputs = ort_session.run(None, {'image': image})
    # Add decoding logic here
    return {"caption": "Sample caption"}

# Save for Docker deployment
with open("main.py", "w") as f:
    f.write(app.get_root_path())